# Build Your AI Agent with MCPs using vLLM, Pydantic AI, and AMD MI300X GPU

Welcome to this hands-on workshop! Throughout this tutorial, we'll leverage AMD GPUs and **Model Context Protocol (MCP)** ,an open standard for exposing LLM tools via API, to deploy powerful language models like Qwen3. Key components:
- 🖥️ **vLLM** for GPU-optimized inference
- 🛠️ **Pydantic-AI** for agent/tool management
- 🔌 **MCP Servers** for pre-built tool integration

You'll learn how to set up your environment, deploy large language models like Qwen3, connect them to real-world tools using MCP, and build a conversational agent capable of reasoning and taking actions.

By the end of this workshop, you’ll have built an AI-powered Airbnb assistant agent—one that can find a place to stay based on your preferences like location, budget, and travel dates.

Let’s dive in!

## Table of Contents

- [Step 1: Launching vLLM Server on AMD GPUs](#step1)
- [Step 2: Installing Dependencies](#step2)
- [Step 3: Create a simple instance of Pydantic-AI Agent](#step3)
- [Step 4: Write a Date/Time Tool for Your Agent](#step4)
- [Step 5: Replace Your Date/time Tool with a MCP server](#step5)
- [Step 6: Turn your agent to an Airbnb finder](#step6)
- [Step 7: Challenge](#step7)

<a id="step1"></a>

## Step 1: Launch a vLLM Server

In this workshop we are going to use [vLLM](https://github.com/vllm-project/vllm) as our inference serving engine. vLLM provides many benefits such as fast model execution, extensive list of supported models, easy to use, and best of all it's open-source. 

### Deploy Qwen3-30B-A3B Model with vLLM




Time to start your vLLM server and creating an end-point for your LLM. Let's open a terminal using your Jupyter server. Then run the following command in this terminal to start the vLLM server:

```bash
VLLM_USE_TRITON_FLASH_ATTN=0 \
vllm serve Qwen/Qwen3-30B-A3B \
    --served-model-name Qwen3-30B-A3B \
    --api-key abc-123 \
    --port 8000 \
    --enable-auto-tool-choice \
    --tool-call-parser hermes \
    --trust-remote-code
```

Open another terminal and monitor the GPU utilization by running this command:

```bash
watch rocm-smi
```

Upon successful launch, your server should be accepting incoming traffic through an OpenAI-compatible API. Let's set some environment variables for our server so we can use throughout this tutorial:

In [1]:
import os

BASE_URL = f"http://localhost:8000/v1"

os.environ["BASE_URL"]    = BASE_URL
os.environ["OPENAI_API_KEY"] = "abc-123"   

print("Config set:", BASE_URL)

Config set: http://localhost:8000/v1


We can verify your model is available at the `BASE_URL` we just set by running the following command.

In [2]:
!curl http://localhost:8000/v1/models -H "Authorization: Bearer $OPENAI_API_KEY"

{"object":"list","data":[{"id":"Qwen3-30B-A3B","object":"model","created":1781629292,"owned_by":"vllm","root":"Qwen/Qwen3-30B-A3B","parent":null,"max_model_len":40960,"permission":[{"id":"modelperm-b3a385e446e25076","object":"model_permission","created":1781629292,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

Congratulations, you now just launched a powerful server that can serve any incoming request and allowing you to build amazing applications. Wasn't that easy?🎉 

<a id="step2"></a>

## Step 2: Installing Dependencies

We are going to use `Pydantic AI`. Let's install the dependencies:

In [3]:
!pip install -q pydantic-ai-slim openai


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip



<a id="step3"></a>

## Step 3: Create a simple instance of Pydantic-AI Agent

Let's start by creating a custom OpenAI Compatible endpoint for our agent. 


In [4]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

provider = OpenAIProvider(
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

agent_model = OpenAIChatModel("Qwen3-30B-A3B", provider=provider)

Let's start by creating an instance the `Agent` class from `pydantic_ai`. 


In [5]:
from pydantic_ai import Agent

agent = Agent(
    model=agent_model
)

It's time to test the agent. `pydantic_ai` provides multiple ways to run `Agent`. You can learn more about it [here](https://ai.pydantic.dev/agents/#running-agents).

In this workshop, we are running in `async` mode. We are going to define a helper function that allows us to quickly test our agent throughout this workshop.

In [6]:
import asyncio
from pydantic_ai.mcp import MCPServerStdio
async def run_async(prompt: str) -> str:
    async with agent.run_mcp_servers():
        result = await agent.run(prompt)
        return result.output


Test the agent by calling this function.

In [7]:
await run_async("What is the capital of France?")

'\n\nThe capital of France is **Paris**. It is a major city in Europe and is known for its iconic landmarks such as the Eiffel Tower, the Louvre Museum, and Notre-Dame Cathedral. Paris serves as the political, economic, and cultural center of France.'

Great! now that we have the basics of creating an agent instance, and connecting it to the model we started serving with vLLM earlier.

<a id="step4"></a>

## Step 4: Write a Date/Time Tool for Your Agent

LLMs naturally rely on their training data to respond to your prompts. Therefore, the agent we just defined fails to answer a factual question that falls outside of it's training knowledge. Let's show this with an example:

In [8]:
await run_async("What’s the date today?")

"\n\nI don't have access to real-time information, so I can't provide the current date. However, I can tell you that my knowledge is current as of October 2023. If you need the latest date, please check a calendar or device clock. Let me know if there's anything else I can assist with!"

It is no surprise that the model failed to answer this question. Now, it's time to power-up your LLM by providing `agent` a function that can get the current date. The process of an LLM triggering a function call is commonly referred to as `Tool Calling` or `Function Calling`. In this workshop we are going to take advantage of `pydantic-ai`'s agent `tools` to provide our agent appropriate tools. First, we need to define a custom tool. Below is how we can define a tool in this framework.

In [9]:
from datetime import datetime
from pydantic_ai import Tool          
@Tool
def get_current_date() -> str:
    """Return the current date/time as an ISO-formatted string."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


Next, we need to provide this tool to our Agent, as this will notify the LLM about the existence of such a tool we just definied. This is simply done by just providing the function signiture of the tool we just defined to our agent constructor. 

In [10]:
agent = Agent(
    model=agent_model,
    tools=[get_current_date],
    system_prompt = (
        "You have access to:\n"
        "   1. get_current_time(params: dict)\n"
        "Use this tool for date/time questions."
    )
)

Let's test the agent.

In [11]:
await run_async("What’s the date today?")

'\n\nThe current date is **June 16, 2026**, and the time is **5:02 PM**. Let me know if you need further details!'

Well done on building an agent with access to real-time data. 



<a id="step5"></a>

## Step 5: Replace Your Date/time Tool with a MCP server

Now that we learned how to create a custom tool and provide the agent access to this tool. Let's now explore a trendy topic of [Model Context Protocol](https://modelcontextprotocol.io/introduction). We are going to explore how we can replace our custom tool with a simple MCP server that can serve our agent and provide similar information.

**Why MCP?** MCP servers provide:
- ✅ Standardized API interfaces
- 🔄 Reusable across projects
- 📦 Pre-built functionality

Let's replace our custom time tool with an official MCP time server:

### Installing Time MCP Server

We are going to start by installing this MCP server:


In [12]:
!pip install -q mcp-server-time


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


Now let's define our time_server:

In [13]:
from pydantic_ai.mcp import MCPServerStdio

time_server = MCPServerStdio(
    "python",
    args=[
        "-m", "mcp_server_time",
        "--local-timezone=America/New_York",
    ],
)

/tmp/ipykernel_78/3040644900.py:3: DeprecationWarning: `MCPServerStdio` is deprecated and will be removed in v2. Use `MCPToolset('path/to/script.py')` for Python scripts, `MCPToolset('script.js')` for Node scripts, or `MCPToolset(fastmcp.client.transports.StdioTransport(command='...', args=[...]))` for arbitrary commands.
  time_server = MCPServerStdio(


Finally, let's modify our agent to remove our previously defined tool, and add this MCP server instead.

In [14]:
agent = Agent(
    model=agent_model,
    mcp_servers=[time_server],
    system_prompt = (
        "You are a helpful agent and you have access to this tool:\n"
        "   get_current_time(params: dict)\n"
        "When the user asks for the current date or time, call get_current_time.\n"
    )
)

Great, let's see if the agent can use the MCP to give us the correct time now.

In [15]:
await run_async("What’s the date today?")

'\n\nThe current date is **June 16, 2026** (Tuesday) in the America/New_York timezone.'


Tadaa! Now you have officially used an MCP server to power-up your agent. In the next section we show how you can your turn many ideas into real working projects by using 100s of free or paid MCP servers available today.



<a id="step6"></a>

## Step 6: Turn your agent to an airbnb finder

As we experience in the last section, MCP servers are really easy to use and they provide a standard way of providing LLMs the tools we need. There are already thousands of MCP servers available for us to use. There are some MCP trackers that you can always use to find out about available servers. Here are some for your reference:
- https://github.com/modelcontextprotocol/servers
- https://mcp.so/

We are going to use npx to launch out next server. Therefore, let's install the required dependencies.

In [16]:
# Install Node.js 20 via NodeSource
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!apt install -y nodejs

2026-06-16 17:02:06 - Installing pre-requisites
Get:1 https://repo.radeon.com/amdgpu/30.30.1/ubuntu jammy InRelease [3185 B]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:3 https://repo.radeon.com/rocm/apt/7.2.2 jammy InRelease [2605 B]          
Get:4 http://archive.ubuntu.com/ubuntu jammy InRelease [270 kB]               m
Get:5 https://repo.radeon.com/amdgpu/30.30.1/ubuntu jammy/main amd64 Packages [1332 B]3m
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]      
Get:8 https://repo.radeon.com/rocm/apt/7.2.2 jammy/main amd64 Packages [71.3 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4024 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1306 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [77.8 kB]
Get:12 http://security.ubuntu.com/ubun

Verify `npm` and `npx` installation:

In [17]:
!node -v && npm -v && npx --version

v20.20.2
10.8.2
10.8.2




In this part of the workshop we are going to build an agent that can help you browse available Airbnbs to book. We can now build on top of what we have so far and add an open-source Airbnb MCP server to our agent. To do so, let's start by defining our Airbnb server.

In [18]:
airbnb_server = MCPServerStdio(
    "npx", args=["-y", "@openbnb/mcp-server-airbnb", "--ignore-robots-txt"]
)

/tmp/ipykernel_78/1537382601.py:1: DeprecationWarning: `MCPServerStdio` is deprecated and will be removed in v2. Use `MCPToolset('path/to/script.py')` for Python scripts, `MCPToolset('script.js')` for Node scripts, or `MCPToolset(fastmcp.client.transports.StdioTransport(command='...', args=[...]))` for arbitrary commands.
  airbnb_server = MCPServerStdio(


Let's update our agent.

In [19]:
system_prompt = """
You have access to three tools:
1. get_current_time(params: dict)
2. airbnb_search(params: dict)
3. airbnb_listing_details(params: dict)
When the user asks for listings, first call get_current_time, then airbnb_search, etc.
"""

agent = Agent(
    model=agent_model,
    mcp_servers=[time_server, airbnb_server],
    system_prompt=system_prompt,
)


Finally, let's try our agent and see if it can browse through Airbnb listings.

In [20]:
await run_async("Find a place to stay in Vancouver for next Sunday for 3 nights for 2 adults?")

'\n\nHere are some Airbnb listings in Vancouver for your stay from June 21 to June 24, 2026 (3 nights):\n\n1. **The Green Home**  \n   - **Price**: $804.20 (was $1,075.55)  \n   - **Rating**: ⭐ 5.0 (103 reviews)  \n   - **Details**: 1 bedroom, 1 queen bed, 1 bath  \n   - [View Listing](https://www.airbnb.com/rooms/1050196677325020460)  \n\n2. **Cozy studio in Mount Pleasant**  \n   - **Price**: $445.94 (was $1,121)  \n   - **Rating**: ⭐ 4.75 (146 reviews)  \n   - **Details**: 2 beds, 1 bath  \n   - [View Listing](https://www.airbnb.com/rooms/717551955250963710)  \n\n3. **Stylish Yaletown Condo**  \n   - **Price**: $761.10  \n   - **Rating**: ⭐ 5.0 (3 reviews)  \n   - **Details**: 2 bedrooms, 2 beds, 1 bath  \n   - [View Listing](https://www.airbnb.com/rooms/1691025131584572464)  \n\n4. **Prime Downtown Location**  \n   - **Price**: $690.35 (was $1,057)  \n   - **Rating**: ⭐ 5.0 (8 reviews)  \n   - **Details**: 1 bedroom, 1 bed, 1 bath  \n   - [View Listing](https://www.airbnb.com/rooms



<a id="step7"></a>

## Step 7: Challenge - Expand the Agent

**Task:** Add weather integration from any available MCP server you can find.

1. Launch weather MCP server
2. Add to agent's tools
3. Make agent suggest best travel dates based on weather

Happy coding! If you encounter issues or have questions, don’t hesitate to ask or raise an issue on our [Github page](https://github.com/ROCm/gpuaidev)!

In [23]:
pip install pandas openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [28]:
import pandas as pd
# Load Excel file
df = pd.read_excel("examples/Service_Contact_Repository_training_data.xlsx", engine="openpyxl")
# Preview
print(df.head())

   Service ID Third Party Name   Service Owner       Latest contact  \
0           1              ABC  Paul Patterson      JohnDoe@ABC.com   
1           2              DEF    Andy Markram      JaneDoe@DEF.com   
2           3              GHI       Ajay Raju    DavidMarx@GHI.com   
3           4              JKL  Anthony Mcleod   MoniqShaha@JKL.com   
4           5              MNO     Monty Rhode  RameshRahul@MNO.com   

  Previous contact  
0       JD@ABC.com  
1       JD@DEF.com  
2       DM@GHI.com  
3       MS@JKL.com  
4       RR@MNO.com  


In [31]:
print(df.columns)

Index(['Service ID', 'Third Party Name', 'Service Owner', 'Latest contact',
       'Previous contact'],
      dtype='str')


In [32]:
# Convert to list of dicts
data = [{"instruction": q, "output": a} for q, a in zip(df["Third Party Name"], df["Latest contact"])]


In [33]:

# Save to JSONL
import json
with open("train.jsonl", "w", encoding="utf-8") as f:
    for row in data:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


In [34]:
#Load dataset for training
from datasets import load_dataset

dataset = load_dataset("json", data_files="train.jsonl")

Generating train split: 0 examples [00:00, ? examples/s]

In [35]:
#Prepare Qwen model
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2-1.5B"  # example model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [39]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# ✅ Tokenization function
def tokenize(batch):
    texts = [instr + "\n" + out for instr, out in zip(batch["instruction"], batch["output"])]
    return tokenizer(texts, truncation=True)

# Apply tokenization
tokenized = dataset.map(tokenize, batched=True)

# ✅ Enable gradient checkpointing to save memory
model.gradient_checkpointing_enable()

# ✅ Data collator for causal language modeling (Qwen is causal LM, not masked LM)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# ✅ Training arguments with memory‑friendly settings
training_args = TrainingArguments(
    output_dir="./qwen-finetuned",
    per_device_train_batch_size=1,       # smaller batch size
    gradient_accumulation_steps=4,       # accumulate gradients to simulate larger batch
    num_train_epochs=3,
    logging_dir="./logs",
    save_strategy="epoch",
    fp16=True,                           # mixed precision (saves memory)
    optim="adamw_torch"                  # efficient optimizer
)

# ✅ Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    data_collator=data_collator
)

# ✅ Start training
trainer.train()


[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: bgemm_internal_cublaslt error: HIPBLAS_STATUS_INTERNAL_ERROR when calling hipblasLtMatmul with transpose_mat1 0 transpose_mat2 0 m 1536 n 8 k 151936 lda 1536 ldb 151936 ldc 1536 abType 2 cType 2 computeType 2 scaleType 0. Will attempt to recover by calling cublas instead. (Triggered internally at /app/pytorch/aten/src/ATen/hip/HIPBlas.cpp:575.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


RuntimeError: CUDA error: HIPBLAS_STATUS_ALLOC_FAILED when calling `hipblasCreate(handle)`